## **Importing Required Libraries**

In [16]:
import pandas as pd
import numpy as np

import nltk
import string

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [15]:
import nltk

nltk.download('punkt')
nltk.download('punk_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Error loading punk_tab: Package 'punk_tab' not found in
[nltk_data]     index
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## **Uploading from Kaggle**

In [3]:
!pip install kaggle
#Upload kaggle API
from google.colab import files

# 1. Upload the kaggle.json file
uploaded = files.upload()

# 2. Setup the Kaggle API directory and permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle (1).json to kaggle (1).json
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [4]:
# Download the dataset using the Kaggle dataset path
!kaggle datasets download -d yufengdev/bbc-fulltext-and-category

# Unzip the downloaded file (-q hides the massive output logs)
!unzip bbc-fulltext-and-category.zip

Dataset URL: https://www.kaggle.com/datasets/yufengdev/bbc-fulltext-and-category
License(s): CC0-1.0
100% 1.83M/1.83M [00:00<00:00, 36.6MB/s]

Archive:  bbc-fulltext-and-category.zip
  inflating: bbc-text.csv            


## **Displaying the rows of dataset**

In [35]:
df = pd.read_csv("bbc-text.csv")
df.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [36]:
df.tail()

,category,text
2220,business,cars pull down us retail figures us retail sal...
2221,politics,kilroy unveils immigration policy ex-chatshow ...
2222,entertainment,rem announce new glasgow concert us band rem h...
2223,politics,how political squabbles snowball it s become c...
2224,sport,souness delight at euro progress boss graeme s...


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  2225 non-null   object
 1   text      2225 non-null   object
dtypes: object(2)
memory usage: 34.9+ KB


In [40]:
df.isnull().sum()

,0
category,0
text,0


In [41]:
# Converting to lower case
df['text'] = df['text'].str.lower()

#Removing punctuation marks
df['text'] = df['text'].str.translate(
    str.maketrans('', '', string.punctuation)
)

#Removing numbers
df['text'] = df['text'].str.replace(r'\d+', '', regex=True)

df['tokens'] = df['text'].apply(lambda x: x.split())

In [42]:
stop_words = set(stopwords.words('english'))

df['tokens'] = df['tokens'].apply(
    lambda words: [w for w in words if w not in stop_words]
)

In [43]:
lemmatizer = WordNetLemmatizer()

df['tokens'] = df['tokens'].apply(
    lambda words: [lemmatizer.lemmatize(w) for w in words]
)

In [44]:
df['clean_text'] = df['tokens'].apply(lambda x: " ".join(x))

In [45]:
X = df['clean_text']
y = df['category']

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [55]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## **Building text classification model**

In [54]:
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [49]:
predictions = model.predict(X_test_tfidf)

## **Categorizing articles automatically**

In [50]:
# Predict categories for all articles
df['Predicted_Category'] = model.predict(vectorizer.transform(df['clean_text']))

# Display the table
df[['category', 'Predicted_Category', 'text']]

,category,Predicted_Category,text
0,tech,tech,tv future in the hands of viewers with home th...
1,business,business,worldcom boss left books alone former worldc...
2,sport,sport,tigers wary of farrell gamble leicester say ...
3,sport,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,entertainment,ocean s twelve raids box office ocean s twelve...
...,...,...,...
2220,business,business,cars pull down us retail figures us retail sal...
2221,politics,politics,kilroy unveils immigration policy exchatshow h...
2222,entertainment,entertainment,rem announce new glasgow concert us band rem h...
2223,politics,politics,how political squabbles snowball it s become c...


## **Evaluating Model Perfomance**

In [57]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.9640449438202248


In [58]:
print(classification_report(y_test, predictions))

               precision    recall  f1-score   support

     business       0.96      0.95      0.96       101
entertainment       1.00      0.89      0.94        81
     politics       0.92      0.99      0.95        83
        sport       0.99      1.00      0.99        98
         tech       0.95      0.99      0.97        82

     accuracy                           0.96       445
    macro avg       0.96      0.96      0.96       445
 weighted avg       0.97      0.96      0.96       445



In [59]:
cm = confusion_matrix(y_test, predictions)

print(cm)

[[96  0  5  0  0]
 [ 3 72  2  0  4]
 [ 1  0 82  0  0]
 [ 0  0  0 98  0]
 [ 0  0  0  1 81]]
